# STab — Transformer Tabular (PyTorch manual) — Telco Churn

Implementação manual de um Transformer para dados tabulares:
- **Feature Tokenizer**: cada feature numérica/binária → projeção linear para `d_model`
- **CLS token**: vetor aprendível agregador
- **TransformerEncoder**: N camadas de self-attention + FFN
- **Cabeça de classificação**: CLS output → Linear(d_model, 2)

Optuna busca `d_model`, `n_heads`, `n_layers`, `dropout`, `lr` (maximiza KS no val).

In [1]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn import metrics as skm
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from model_utils import load_data, compute_metrics, save_results, print_scores

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 1. Carregar dados

In [2]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
INPUT_SIZE = X_train.shape[1]  # 22 features

def to_tensors(X, y):
    return (torch.FloatTensor(X.values).to(device),
            torch.LongTensor(y.values.astype(int)).to(device))

Xtr, ytr = to_tensors(X_train, y_train)
Xvl, yvl = to_tensors(X_val, y_val)
Xte, yte = to_tensors(X_test, y_test)
y_val_np  = y_val.values.astype(int)
y_test_np = y_test.values.astype(int)

Train: (7242, 22)  Val: (1057, 22)  Test: (1057, 22)


## 2. Definição do STab

In [3]:
class STabClassifier(nn.Module):
    """
    Feature Tokenizer + Transformer Encoder + CLS classification head.
    Each of the F input features is projected to d_model via a separate linear.
    A learnable [CLS] token is prepended → sequence length = F+1.
    """
    def __init__(self, n_features, d_model, n_heads, n_layers, dim_feedforward, dropout):
        super().__init__()
        # Per-feature projection: (batch, F) → (batch, F, d_model)
        self.feature_proj = nn.Linear(1, d_model)
        # Learnable CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,   # Pre-LN (more stable)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        # Classification head
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 2),
        )

    def forward(self, x):
        # x: (B, F)
        B = x.size(0)
        # Project each feature independently: (B, F, 1) → (B, F, d_model)
        tokens = self.feature_proj(x.unsqueeze(-1))
        # Prepend CLS: (B, F+1, d_model)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        # Transformer
        out = self.transformer(tokens)
        # Use CLS output for classification
        return self.head(out[:, 0, :])


def eval_ks(model, X, y_np):
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(X), dim=1)[:, 1].cpu().numpy()
    fpr, tpr, _ = skm.roc_curve(y_np, probs)
    return float(np.max(tpr - fpr))

def make_loader(Xt, yt, batch_size, shuffle=True):
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

## 3. Optuna HPO (20 trials, maximiza KS no val)

In [4]:
def objective(trial):
    d_model  = trial.suggest_categorical('d_model', [32, 64, 128])
    n_heads  = trial.suggest_categorical('n_heads', [2, 4])
    # n_heads must divide d_model
    if d_model % n_heads != 0:
        raise optuna.exceptions.TrialPruned()
    n_layers        = trial.suggest_int('n_layers', 1, 3)
    ff_mult         = trial.suggest_categorical('ff_mult', [2, 4])
    dim_feedforward = d_model * ff_mult
    dropout         = trial.suggest_float('dropout', 0.0, 0.3)
    lr              = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    batch_size      = trial.suggest_categorical('batch_size', [32, 64])

    model = STabClassifier(INPUT_SIZE, d_model, n_heads, n_layers,
                           dim_feedforward, dropout).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    loader    = make_loader(Xtr, ytr, batch_size)

    best_ks, patience_counter = 0.0, 0
    for epoch in range(30):
        model.train()
        for Xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(Xb), yb).backward()
            optimizer.step()
        ks = eval_ks(model, Xvl, y_val_np)
        if ks > best_ks:
            best_ks = ks; patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5: break
    return best_ks

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(objective, n_trials=20, show_progress_bar=True)
print(f'Best KS (val): {study.best_value:.4f}')
print('Best params:', study.best_params)

  0%|          | 0/20 [00:00<?, ?it/s]

Best KS (val): 0.4246
Best params: {'d_model': 128, 'n_heads': 4, 'n_layers': 3, 'ff_mult': 2, 'dropout': 0.23138735707139713, 'lr': 0.00022455189152106224, 'batch_size': 32}


## 4. Retreinar com melhores parâmetros

In [5]:
bp = study.best_params
best_model = STabClassifier(
    INPUT_SIZE,
    d_model=bp['d_model'],
    n_heads=bp['n_heads'],
    n_layers=bp['n_layers'],
    dim_feedforward=bp['d_model'] * bp['ff_mult'],
    dropout=bp['dropout'],
).to(device)
optimizer = optim.AdamW(best_model.parameters(), lr=bp['lr'], weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
loader    = make_loader(Xtr, ytr, bp['batch_size'])

best_ks, best_state, patience_counter = 0.0, None, 0
for epoch in range(50):
    best_model.train()
    epoch_loss = 0.0
    for Xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(best_model(Xb), yb)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    ks = eval_ks(best_model, Xvl, y_val_np)
    if ks > best_ks:
        best_ks = ks
        best_state = {k: v.clone() for k, v in best_model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 10:
            print(f'Early stop @ epoch {epoch+1}'); break
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}  loss={epoch_loss/len(loader):.4f}  val_KS={ks:.4f}')

best_model.load_state_dict(best_state)
print(f'Best val KS: {best_ks:.4f}')

Epoch  10  loss=0.5638  val_KS=0.3968
Epoch  20  loss=0.5510  val_KS=0.4039
Early stop @ epoch 26
Best val KS: 0.4231


## 5. Avaliar no teste + salvar artefatos

In [6]:
best_model.eval()
with torch.no_grad():
    logits = best_model(Xte)
    probs  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
    preds  = logits.argmax(dim=1).cpu().numpy()

scores = compute_metrics(y_test_np, preds, probs)
print('Scores no teste:')
print_scores(scores)

os.makedirs('results/stab', exist_ok=True)
torch.save({'state_dict': best_state, 'best_params': bp,
            'n_features': INPUT_SIZE, 'scores': scores},
           'results/stab/model.pt')

save_results('stab', y_test_np, preds, probs, scores)

Scores no teste:
  accuracy               0.7332
  balanced_accuracy      0.7055
  precision              0.4973
  recall                 0.6464
  f1                     0.5621
  auroc                  0.7981
  ks                     0.4612
Saved → results/stab
